# Session 8 — Tabular Q-learning on the Electronics desk

**Salih Tutun, PhD** · DAT 5668 · Module 4 · Practice · Explain → practice together

Adapted from Fall 2025 Lab 4 (inventory Q-learning). Tonight we stay **tabular** (Q1–Q3). DQN waits for Session 9.

## How to use this notebook

| Cue | Meaning |
| --- | --- |
| **RUN WITH ME** | Code ready. Run. Do not edit. |
| **TRY THIS** | Type `<<< FILL` / `TODO`. Re-run. Check. |
| **CHANGE ME** | Change α / ε. Re-run. **PAIR 20s** discuss. |

![Q-table is decide](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-q-decide.png)
![Q-learning update](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/q-update.png)

*All diagrams load from GitHub raw URLs.*


# Part I — Topic

## Same MDP pieces, no P(demand)

Session 7 needed known `P(d)` for Bellman.  
Session 8 learns **Q(s,a)** from Electronics days — model-free.

```text
Q(s,a) ← Q(s,a) + α [ r + γ max_a' Q(s',a') − Q(s,a) ]
```

ε-greedy: with probability ε pick a random action; otherwise `argmax Q(s,·)`.

**Tonight (from Lab 4 Wednesday):**
1. Toy 3-state Q-learning (hand rewards)  
2. Dataset-integrated states from real Electronics inventory  
3. Optional: finer bins  

**JUMP BACK** until I say OPEN.


---
# Part II — Practice


## Stop 0 — Imports + data path


In [ ]:
# STOP 0 — RUN WITH ME · imports

from pathlib import Path
from collections import Counter
import csv
import random
import statistics

import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

DRIVE_DATA_FOLDER = "/content/drive/MyDrive/DAT5668/data"
PRODUCT = "Electronics"
RNG = random.Random(7)


In [ ]:
# STOP 1a — RUN WITH ME · course_data

def course_data(name: str) -> Path:
    # Prefer Drive, then local folders, then GitHub download (Colab-safe)
    candidates = []
    if DRIVE_DATA_FOLDER:
        candidates.append(Path(DRIVE_DATA_FOLDER) / name)
    for base in (
        Path("data"),
        Path("../../Drive-Folder-for-course/DAT5668/data"),
        Path("../Drive-Folder-for-course/DAT5668/data"),
        Path("../../../Drive-Folder-for-course/DAT5668/data"),
        Path("../../assets/data"),
        Path("assets/data"),
        Path("/content/data"),
        Path("../data"),
    ):
        candidates.append(base / name)
    for path in candidates:
        if path.exists():
            return path
    if name in {"inventory_dataset_full.csv", "inventory_dataset.csv"}:
        import urllib.request
        dest = Path("/content/data") if Path("/content").exists() else Path("data")
        dest.mkdir(parents=True, exist_ok=True)
        out = dest / name
        url = "https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/data/" + name
        print("Downloading", name, "from GitHub…")
        urllib.request.urlretrieve(url, out)
        return out
    raise FileNotFoundError("Could not find " + name)


## Stop 1 — Load Electronics desk (RUN WITH ME)


In [ ]:
# STOP 1 — RUN WITH ME · load Electronics

def load_electronics_desk():
    rows = []
    with course_data("inventory_dataset_full.csv").open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            if (row.get("Product") or "").strip() != PRODUCT:
                continue
            rows.append({
                "Date": row["Date"],
                "Current_Inventory": int(float(row["Current_Inventory"])),
                "Demand": int(float(row["Demand"])),
                "Action_Taken": (row.get("Action_Taken") or "").strip(),
                "Profit": float(row.get("Profit") or 0),
            })
    if not rows:
        raise ValueError("No Electronics rows")
    rows.sort(key=lambda r: r["Date"])
    return rows


ROWS = load_electronics_desk()
N_DAYS = len(ROWS)
AVG_DEMAND = statistics.mean(r["Demand"] for r in ROWS)
print("Electronics days:", N_DAYS, "| avg demand:", round(AVG_DEMAND, 1))
print("Actions in log:", dict(Counter(r["Action_Taken"] for r in ROWS)))


## Stop 2 — Warm-up: toy 3-state Q-learning (Lab 4 Q1)

States: Low / Medium / High. Actions: Order / Maintain / Reduce.  
Next state is simplified (random) so you see the update before we wire the real log.


In [ ]:
# STOP 2a — RUN WITH ME · toy Q-learning

STATES = ["Low Inventory", "Medium Inventory", "High Inventory"]
ACTIONS = ["Order", "Maintain", "Reduce"]

REWARDS = {
    ("Low Inventory", "Order"): 10,
    ("Low Inventory", "Maintain"): -5,
    ("Low Inventory", "Reduce"): -10,
    ("Medium Inventory", "Order"): 2,
    ("Medium Inventory", "Maintain"): 5,
    ("Medium Inventory", "Reduce"): 2,
    ("High Inventory", "Order"): -10,
    ("High Inventory", "Maintain"): -5,
    ("High Inventory", "Reduce"): 10,
}

ALPHA = 0.1
EPSILON = 0.1
GAMMA = 0.9
N_EPISODES = 100

Q = {s: {a: 0.0 for a in ACTIONS} for s in STATES}
tracked = []

for _ in range(N_EPISODES):
    state = RNG.choice(STATES)
    for _step in range(10):
        if RNG.random() < EPSILON:
            action = RNG.choice(ACTIONS)
        else:
            action = max(Q[state], key=Q[state].get)
        reward = REWARDS[(state, action)]
        next_state = RNG.choice(STATES)
        old = Q[state][action]
        nxt = max(Q[next_state].values())
        Q[state][action] = old + ALPHA * (reward + GAMMA * nxt - old)
        if state == "Low Inventory" and action == "Order":
            tracked.append(Q[state][action])
        state = next_state

print("Toy Q-table:")
for s in STATES:
    print(" ", s, {a: round(Q[s][a], 2) for a in ACTIONS})
policy = {s: max(Q[s], key=Q[s].get) for s in STATES}
print("Toy policy:", policy)

if plt is not None and tracked:
    fig, ax = plt.subplots(figsize=(5, 2.8))
    ax.plot(tracked, color="#2F5D50")
    ax.set_title("Toy Q(Low, Order) convergence")
    ax.set_xlabel("updates")
    fig.tight_layout()
    plt.show()


### 2b — CHANGE ME · α and ε on the toy desk

**PAIR 20s.** Change learning rate / exploration. Re-run Stop 2a after editing the knobs below…  
Better: change knobs in the next cell and re-train.


In [ ]:
# STOP 2b — CHANGE ME · then re-run training snippet

ALPHA = 0.1    # <<< CHANGE ME: try 0.05 / 0.3 / 1.0
EPSILON = 0.1  # <<< CHANGE ME: try 0.0 / 0.5
GAMMA = 0.9

Q2 = {s: {a: 0.0 for a in ACTIONS} for s in STATES}
tracked2 = []
for _ in range(N_EPISODES):
    state = RNG.choice(STATES)
    for _step in range(10):
        if RNG.random() < EPSILON:
            action = RNG.choice(ACTIONS)
        else:
            action = max(Q2[state], key=Q2[state].get)
        reward = REWARDS[(state, action)]
        next_state = RNG.choice(STATES)
        old = Q2[state][action]
        Q2[state][action] = old + ALPHA * (reward + GAMMA * max(Q2[next_state].values()) - old)
        if state == "Low Inventory" and action == "Order":
            tracked2.append(Q2[state][action])
        state = next_state

print("alpha", ALPHA, "epsilon", EPSILON)
print("policy:", {s: max(Q2[s], key=Q2[s].get) for s in STATES})
if plt is not None and tracked2:
    fig, ax = plt.subplots(figsize=(5, 2.8))
    ax.plot(tracked2, color="#C45C26")
    ax.set_title(f"Q(Low,Order) α={ALPHA} ε={EPSILON}")
    fig.tight_layout()
    plt.show()


## Stop 3 — States from the real Electronics log (Lab 4 Q2)

Map `Current_Inventory / avg Demand` → Low / Medium / High (same idea as Session 6 thresholds).


In [ ]:
# STOP 3a — RUN WITH ME · categorize real days

def categorize_inventory(inv, avg_demand, low_cut=0.5, high_cut=2.0):
    ratio = float(inv) / max(avg_demand, 1e-9)
    if ratio < low_cut:
        return "Low Inventory"
    if ratio <= high_cut:
        return "Medium Inventory"
    return "High Inventory"


for r in ROWS:
    r["State"] = categorize_inventory(r["Current_Inventory"], AVG_DEMAND)

print("State mix:", dict(Counter(r["State"] for r in ROWS)))
if plt is not None:
    counts = Counter(r["State"] for r in ROWS)
    fig, ax = plt.subplots(figsize=(4.5, 2.8))
    labels = ["Low Inventory", "Medium Inventory", "High Inventory"]
    ax.bar(labels, [counts[k] for k in labels], color=["#C45C26", "#2F5D50", "#1F4E79"])
    ax.set_title("Electronics state mix")
    fig.tight_layout()
    plt.show()


## Stop 4 — YOU WRITE the ε-greedy + Q update (Lab 4 core)

Walk consecutive Electronics days. Reward table encodes inventory sense (Order when Low, Reduce when High).

Complete the two `TODO` lines. Run once after you fill.


In [ ]:
# STOP 4 — TRY THIS · Q-learning on real day sequence

STATES = ["Low Inventory", "Medium Inventory", "High Inventory"]
ACTIONS = ["Order", "Maintain", "Reduce"]

REWARDS_DATA = {
    ("Low Inventory", "Order"): 2,
    ("Low Inventory", "Maintain"): -1,
    ("Low Inventory", "Reduce"): -2,
    ("Medium Inventory", "Order"): 0,
    ("Medium Inventory", "Maintain"): 1,
    ("Medium Inventory", "Reduce"): 0,
    ("High Inventory", "Order"): -2,
    ("High Inventory", "Maintain"): -1,
    ("High Inventory", "Reduce"): 2,
}

alpha = 0.3
epsilon = 0.5   # <<< CHANGE ME after first PASS
gamma = 0.7
n_episodes = 50

Q = {s: {a: 0.0 for a in ACTIONS} for s in STATES}
tracked = []


def epsilon_greedy(state, Q, epsilon):
    # TODO: with prob epsilon return random action; else argmax Q[state]
    raise NotImplementedError("TODO: write epsilon_greedy")


def q_update(Q, state, action, reward, next_state, alpha, gamma):
    # TODO: Q[s,a] ← Q[s,a] + α (r + γ max Q[s'] − Q[s,a])
    raise NotImplementedError("TODO: write q_update")


# --- baseline: leave TODOs broken once so you see the failure, then fill ---
try:
    for _ in range(n_episodes):
        for i in range(len(ROWS) - 1):
            state = ROWS[i]["State"]
            action = epsilon_greedy(state, Q, epsilon)
            reward = REWARDS_DATA[(state, action)]
            next_state = ROWS[i + 1]["State"]
            q_update(Q, state, action, reward, next_state, alpha, gamma)
            if state == "Low Inventory" and action == "Order":
                tracked.append(Q[state][action])
    print("Q-table:")
    for s in STATES:
        print(" ", s, {a: round(Q[s][a], 3) for a in ACTIONS})
    print("Policy:", {s: max(Q[s], key=Q[s].get) for s in STATES})
    print("CHECK 4 — PASS")
    if plt is not None and tracked:
        fig, ax = plt.subplots(figsize=(5, 2.8))
        ax.plot(tracked[:: max(1, len(tracked)//200)], color="#1F4E79")
        ax.set_title("Q(Low, Order) on Electronics days")
        fig.tight_layout()
        plt.show()
except NotImplementedError as err:
    print("NOT YET —", err)
    print("Fill epsilon_greedy and q_update, then re-run.")


### Hint (only if stuck)

```python
def epsilon_greedy(state, Q, epsilon):
    if RNG.random() < epsilon:
        return RNG.choice(ACTIONS)
    return max(Q[state], key=Q[state].get)

def q_update(Q, state, action, reward, next_state, alpha, gamma):
    old = Q[state][action]
    nxt = max(Q[next_state].values())
    Q[state][action] = old + alpha * (reward + gamma * nxt - old)
```


## Stop 5 — Optional stretch: 10 inventory bins (Lab 4 Q3)

**CHANGE ME** `NUM_BINS` if you finish early. Coarser = fewer states; finer = more sample-hungry.


In [ ]:
# STOP 5 — CHANGE ME · discretized bins Q-learning

NUM_BINS = 10  # <<< CHANGE ME: try 5 or 15
inv_vals = [r["Current_Inventory"] for r in ROWS]
edges = np.linspace(min(inv_vals), max(inv_vals), num=NUM_BINS)


def to_bin(inv):
    return int(np.digitize(inv, edges))


def bin_reward(state, action):
    if state < 3 and action == "Order":
        return 2
    if 3 <= state < 7 and action == "Maintain":
        return 1
    if state >= 7 and action == "Reduce":
        return 2
    return -1


bin_states = range(NUM_BINS + 1)
Qb = {s: {a: 0.0 for a in ACTIONS} for s in bin_states}
alpha_b, eps_b, gamma_b = 0.3, 0.5, 0.8

for _ in range(30):
    for i in range(len(ROWS) - 1):
        s = to_bin(ROWS[i]["Current_Inventory"])
        if RNG.random() < eps_b:
            a = RNG.choice(ACTIONS)
        else:
            a = max(Qb[s], key=Qb[s].get)
        r = bin_reward(s, a)
        sp = to_bin(ROWS[i + 1]["Current_Inventory"])
        old = Qb[s][a]
        Qb[s][a] = old + alpha_b * (r + gamma_b * max(Qb[sp].values()) - old)

policy_b = {s: max(Qb[s], key=Qb[s].get) for s in bin_states}
print("bins", NUM_BINS, "sample policy bins 0, mid, high:",
      policy_b[0], policy_b[NUM_BINS // 2], policy_b[NUM_BINS])


## Stop 6 — Diagnostics (discuss)

| Change | What you should watch |
| --- | --- |
| `epsilon=0` | Table can freeze on a lucky first action |
| `alpha=1` | Tracks the last sample; oscillates |
| Reward that never punishes Reduce-when-Low | Policy starves the shelf |

Would you put this Q-table on a real SKU tomorrow? Name the **human override** (Session 6 memo).


## Stop 7 — Framework line (TRY THIS)


In [ ]:
# STOP 7 — TRY THIS

framework_family = ""  # <<< FILL: LLM / RL / hybrid
framework_line = ""    # <<< FILL: one sentence
print("Family:", framework_family or "(FILL)")
print("I will keep:", framework_line or "(FILL)")
if not framework_family or not framework_line:
    print("NOT YET")
else:
    print("CHECK 7 — recorded")


## Raise a hand — Session 8 complete

1. [ ] Toy Q policy discussed  
2. [ ] `epsilon_greedy` + `q_update` PASS on Electronics days  
3. [ ] Changed α or ε and said what moved  
4. [ ] Can name a human override for a bad Q row  
